In [ ]:

from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql import SparkSession
import pandas as pd
file_path = "/Workspace/Shared/DevOps_Assignment/Sales_Data.csv"
df = pd.read_csv(file_path)
print("Raw Data Sample:")
display(df.head())

In [ ]:
display(len(df))

In [ ]:
spark.sql("CREATE CATALOG IF NOT EXISTS sd")
spark.sql("CREATE SCHEMA IF NOT EXISTS sd.sales_data_br")
(spark.createDataFrame(df)
.write                                             
.mode("overwrite")                                 
.option("overwriteSchema", "true")
.saveAsTable("sd.sales_data_br.sales_data")) 


In [ ]:
print(f"Ingested {len(df)} rows into sd.sales_data_br.")

In [ ]:
%sql
SELECT * FROM sd.sales_data_br.sales_data

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS sd.sales_data_silver

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS sd.sales_data_silver.sales_data_fact(
  TransactionID STRING,
  `Date` DATE,
  DateTime TIMESTAMP,
  CustomerID STRING,
  ProductID STRING,
  Quantity INT,
  UnitPrice FLOAT,
  Discount FLOAT,
  TotalPrice FLOAT,
  PaymentMethod STRING,
  SalespersonID STRING,
  `Region` STRING)

In [ ]:
%sql
--Constraints for the Sales Data Table--
ALTER TABLE sd.sales_data_silver.sales_data_fact ADD CONSTRAINT TransactionIDNotNull CHECK (TransactionID IS NOT NULL);
    
ALTER TABLE sd.sales_data_silver.sales_data_fact ADD CONSTRAINT CustomerIDNotNul CHECK (CustomerID IS NOT NULL);
    
ALTER TABLE sd.sales_data_silver.sales_data_fact ADD CONSTRAINT ProductIDNotNul CHECK (ProductID IS NOT NULL);

ALTER TABLE sd.sales_data_silver.sales_data_fact ADD CONSTRAINT QuantityPositive CHECK (Quantity > 0);
    
ALTER TABLE sd.sales_data_silver.sales_data_fact ADD CONSTRAINT UnitPriceNonNegative CHECK (UnitPrice >= 0);
    
ALTER TABLE sd.sales_data_silver.sales_data_fact ADD CONSTRAINT TotalPriceNonNegative CHECK (TotalPrice >= 0);
    
ALTER TABLE sd.sales_data_silver.sales_data_fact ADD CONSTRAINT DiscountAppliedNonNegative CHECK (Discount >= 0 AND Discount <= 1);



In [ ]:
%sql
INSERT OVERWRITE sd.sales_data_silver.sales_data_fact
SELECT
  BS.TransactionID AS TransactionID,
  to_date(BS.Date, 'M/d/yyyy') AS Date,
  to_timestamp(concat(BS.Date, ' ', BS.Time), 'M/d/yyyy H:mm:ss') AS DateTime,
  BS.CustomerID AS CustomerID,
  BS.ProductID AS ProductID,
  BS.Quantity AS Quantity,
  BS.UnitPrice AS Price,
 CAST(REGEXP_REPLACE(BS.DiscountApplied, '%', '') AS DOUBLE) / 100  AS Discount,
  BS.TotalPrice AS TotalPrice,
  BS.PaymentMethod AS PaymentMethod,
  BS.SalespersonID AS SalespersonID,
  BS.Region AS `Region`
FROM sd.sales_data_br.sales_data AS BS

In [ ]:

%sql
CREATE TABLE IF NOT EXISTS sd.sales_data_silver.products
(ProductID STRING,
 ProductName STRING,
 Category STRING
)

In [ ]:
# %sql
# --Constraints for the Products Table--
# ALTER TABLE sd.sales_data_silver.products ADD CONSTRAINT ProductIDNotNull CHECK (ProductID IS NOT NULL);


In [ ]:
%sql
INSERT OVERWRITE sd.sales_data_silver.products
SELECT DISTINCT
  BS.ProductID AS ProductID,
  BS.ProductName AS ProductName,
  BS.Category AS Category
FROM sd.sales_data_br.sales_data AS BS

In [ ]:

%sql
CREATE TABLE IF NOT EXISTS sd.sales_data_silver.customers(
    CustomerID STRING,
    CustomerAge INT,
    CustomerGender STRING
)

In [ ]:
# %sql
# --Constraints for the Customers Table--
# ALTER TABLE sd.sales_data_silver.customers ADD CONSTRAINT CustomerIDNotNull CHECK (CustomerID IS NOT NULL);



In [ ]:
%sql
INSERT OVERWRITE sd.sales_data_silver.customers
WITH min_year AS (
  SELECT MIN(YEAR(to_date(Date, 'M/d/yyyy'))) AS Year,
  CustomerID AS CustomerID
  FROM sd.sales_data_br.sales_data
  GROUP BY CustomerID
)
SELECT DISTINCT
  BS.CustomerID AS CustomerID,
  BS.CustomerAge + YEAR(CURRENT_DATE()) - min_year.Year AS CustomerAge,
  BS.CustomerGender AS CustomerGender
FROM sd.sales_data_br.sales_data AS BS,min_year
WHERE BS.CustomerID = min_year.CustomerID


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS sd.sales_data_gold

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS sd.sales_data_gold.regional_sales_summary (
  Region STRING,
  TotalRevenue DECIMAL(15,2),
  TotalQuantitySold INT,
  TotalTransactions INT,
  AverageOrderValue DECIMAL(10,2),
  TotalDiscountGiven DECIMAL(15,2),
  AverageDiscount DECIMAL(5,4),
  UniqueCustomers INT,
  UniqueProducts INT
)

In [ ]:
%sql
INSERT OVERWRITE sd.sales_data_gold.regional_sales_summary
SELECT
  f.Region,
  ROUND(SUM(f.TotalPrice), 2) AS TotalRevenue,
  SUM(f.Quantity) AS TotalQuantitySold,
  COUNT(DISTINCT f.TransactionID) AS TotalTransactions,
  ROUND(AVG(f.TotalPrice), 2) AS AverageOrderValue,
  ROUND(SUM(f.TotalPrice * f.Discount), 2) AS TotalDiscountGiven,
  ROUND(AVG(f.Discount), 4) AS AverageDiscount,
  COUNT(DISTINCT f.CustomerID) AS UniqueCustomers,
  COUNT(DISTINCT f.ProductID) AS UniqueProducts
FROM sd.sales_data_silver.sales_data_fact f
GROUP BY f.Region
ORDER BY TotalRevenue DESC

In [ ]:
%sql
SELECT * FROM sd.sales_data_gold.regional_sales_summary
ORDER BY TotalRevenue DESC

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS sd.sales_data_gold.customer_sales_summary (
  CustomerID STRING,
  CustomerAge INT,
  CustomerGender STRING,
  TotalRevenue DECIMAL(15,2),
  TotalQuantityPurchased INT,
  TotalTransactions INT,
  AverageOrderValue DECIMAL(10,2),
  TotalDiscountReceived DECIMAL(15,2),
  AverageDiscount DECIMAL(5,4),
  UniqueProductsPurchased INT,
  FirstPurchaseDate DATE,
  LastPurchaseDate DATE,
  PreferredPaymentMethod STRING,
  PreferredRegion STRING
)

In [ ]:
%sql
INSERT OVERWRITE sd.sales_data_gold.customer_sales_summary
WITH customer_aggregates AS (
  SELECT
    f.CustomerID,
    ROUND(SUM(f.TotalPrice), 2) AS TotalRevenue,
    SUM(f.Quantity) AS TotalQuantityPurchased,
    COUNT(DISTINCT f.TransactionID) AS TotalTransactions,
    ROUND(AVG(f.TotalPrice), 2) AS AverageOrderValue,
    ROUND(SUM(f.TotalPrice * f.Discount), 2) AS TotalDiscountReceived,
    ROUND(AVG(f.Discount), 4) AS AverageDiscount,
    COUNT(DISTINCT f.ProductID) AS UniqueProductsPurchased,
    MIN(f.Date) AS FirstPurchaseDate,
    MAX(f.Date) AS LastPurchaseDate
  FROM sd.sales_data_silver.sales_data_fact f
  GROUP BY f.CustomerID
),
preferred_payment AS (
  SELECT 
    CustomerID,
    PaymentMethod,
    ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY COUNT(*) DESC) AS rn
  FROM sd.sales_data_silver.sales_data_fact
  GROUP BY CustomerID, PaymentMethod
),
preferred_region AS (
  SELECT 
    CustomerID,
    Region,
    ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY COUNT(*) DESC) AS rn
  FROM sd.sales_data_silver.sales_data_fact
  GROUP BY CustomerID, Region
)
SELECT
  ca.CustomerID,
  c.CustomerAge,
  c.CustomerGender,
  ca.TotalRevenue,
  ca.TotalQuantityPurchased,
  ca.TotalTransactions,
  ca.AverageOrderValue,
  ca.TotalDiscountReceived,
  ca.AverageDiscount,
  ca.UniqueProductsPurchased,
  ca.FirstPurchaseDate,
  ca.LastPurchaseDate,
  pp.PaymentMethod AS PreferredPaymentMethod,
  pr.Region AS PreferredRegion
FROM customer_aggregates ca
LEFT JOIN sd.sales_data_silver.customers c ON ca.CustomerID = c.CustomerID
LEFT JOIN preferred_payment pp ON ca.CustomerID = pp.CustomerID AND pp.rn = 1
LEFT JOIN preferred_region pr ON ca.CustomerID = pr.CustomerID AND pr.rn = 1
ORDER BY TotalRevenue DESC

In [ ]:
%sql
SELECT * FROM sd.sales_data_gold.customer_sales_summary
ORDER BY TotalRevenue DESC
LIMIT 20

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS sd.sales_data_gold.product_sales_summary (
  ProductID STRING,
  ProductName STRING,
  Category STRING,
  TotalRevenue DECIMAL(15,2),
  TotalQuantitySold INT,
  TotalTransactions INT,
  AverageUnitPrice DECIMAL(10,2),
  AverageDiscount DECIMAL(5,4),
  TotalDiscountGiven DECIMAL(15,2),
  UniqueCustomers INT,
  FirstSaleDate DATE,
  LastSaleDate DATE
)

In [ ]:
%sql
INSERT OVERWRITE sd.sales_data_gold.product_sales_summary
SELECT
  p.ProductID,
  p.ProductName,
  p.Category,
  ROUND(SUM(f.TotalPrice), 2) AS TotalRevenue,
  SUM(f.Quantity) AS TotalQuantitySold,
  COUNT(DISTINCT f.TransactionID) AS TotalTransactions,
  ROUND(AVG(f.UnitPrice), 2) AS AverageUnitPrice,
  ROUND(AVG(f.Discount), 4) AS AverageDiscount,
  ROUND(SUM(f.TotalPrice * f.Discount), 2) AS TotalDiscountGiven,
  COUNT(DISTINCT f.CustomerID) AS UniqueCustomers,
  MIN(f.Date) AS FirstSaleDate,
  MAX(f.Date) AS LastSaleDate
FROM sd.sales_data_silver.sales_data_fact f
LEFT JOIN sd.sales_data_silver.products p ON f.ProductID = p.ProductID
GROUP BY p.ProductID, p.ProductName, p.Category
ORDER BY TotalRevenue DESC

In [ ]:
%sql
SELECT * FROM sd.sales_data_gold.product_sales_summary
ORDER BY TotalRevenue DESC
LIMIT 20